<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# LightGBM Learning-to-Rank with LambdaRank on MovieLens

This notebook demonstrates how to build a learning-to-rank pipeline using `lgb.LGBMRanker` with the `lambdarank` objective, which optimises NDCG directly.

We use the **MovieLens 1M** dataset: each user is treated as a *query*, and their rated movies are the *candidate items*. Labels are **binarized** using a user-relative threshold (rating > user's mean → 1, else → 0). The evaluation pool includes **sampled negative items** (movies the user never rated) to produce realistic NDCG scores — without negatives, every item has relevance > 0 and the model trivially achieves high NDCG.

```python
import lightgbm as lgb
from lightgbm import LGBMRanker

ranker = LGBMRanker(n_estimators=NUM_BOOST_ROUND, **PARAMS)

ranker.fit(
    train_x,
    train_y.reshape(-1),
    group=train_group,
    eval_set=[(valid_x, valid_y.reshape(-1))],
    eval_group=[valid_group],
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)],
)
ranker.booster_.save_model("model.lgb")

booster = lgb.Booster(model_file="model.lgb")
scores  = booster.predict(test_x)
```

## 0. Imports and Parameters

In [1]:
import os
import sys
from tempfile import TemporaryDirectory

import lightgbm as lgb
import numpy as np
import pandas as pd
from lightgbm import LGBMRanker
from recommenders.datasets import movielens
from recommenders.datasets.pandas_df_utils import negative_feedback_sampler
from recommenders.datasets.python_splitters import python_chrono_split
from recommenders.evaluation.python_evaluation import map, ndcg_at_k
from recommenders.models.lightgbm.lightgbm_utils import NumEncoder
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"LightGBM version: {lgb.__version__}")

System version: 3.11.0 (main, Apr  2 2025, 13:12:15) [Clang 17.0.0 (clang-1700.0.13.3)]
LightGBM version: 4.6.0


/Users/wooklee/.venvs/recommenders/lib/python3.11/site-packages/pandera/_pandas_deprecated.py:157: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [2]:
# Data settings
MOVIELENS_DATA_SIZE = "10m"
USER_COL = "userID"
ITEM_COL = "itemID"
RATING_COL = "rating"
LABEL_COL = "label"  # 1 if rating > user's personal mean, else 0
SEED = 42

# Engineered feature columns
NUME_COLS = [
    "user_mean_rating",
    "user_rating_count",
    "user_rating_std",
    "item_mean_rating",
    "item_rating_count",
    "item_rating_std",
    "user_genre_mean_rating",
    "user_genre_count",
    "release_year",
]
CATE_COLS = ["main_genre"]

# lgb.LGBMRanker parameters (lambdarank with binary relevance labels)
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "num_leaves": 64,
    "learning_rate": 0.01,
    "colsample_bytree": 0.8,
    "n_jobs": 4,
}
NUM_BOOST_ROUND = 10000
EARLY_STOPPING_ROUNDS = 100
N_NEG_TRAIN = 100
N_NEG_VALID = 100
N_NEG_EVAL = 100

## 1. Data Preparation

We use **MovieLens 1M**, which provides real user–item interactions with ratings 1–5.

**Learn To Rank framing:**
- **Query** = user (each user represents a search session)
- **Documents** = movies the user has rated
- **Relevance** = binary label: 1 if the rating exceeds the user's personal mean rating, else 0

Using a **user-relative threshold** (instead of a fixed cut-off like ≥ 4) means the label captures whether a movie was above-average *for that user*, which is a better signal of personal preference and keeps the positive rate balanced across users with different rating tendencies.

Binarizing labels is necessary for a realistic NDCG evaluation: if every item in the pool has relevance > 0 (raw ratings 1–5), even a random ordering achieves high NDCG because there is no "irrelevant junk" to penalise.

**Feature engineering** (computed from training data only to avoid leakage):

| Feature | Description |
|---|---|
| `user_mean_rating` | Average rating given by this user |
| `user_rating_count` | Number of ratings by this user |
| `user_rating_std` | Std-dev of ratings by this user (rating consistency) |
| `item_mean_rating` | Average rating received by this movie |
| `item_rating_count` | Number of ratings for this movie (popularity) |
| `item_rating_std` | Std-dev of ratings for this movie (polarisation) |
| `user_genre_mean_rating` | User's average rating for this movie's primary genre |
| `user_genre_count` | Number of movies this user has rated in this genre |
| `release_year` | Release year extracted from the movie title |
| `main_genre` | Primary genre of the movie (LightGBM native categorical) |

**Query groups** are formed by grouping interactions by `userID` — no simulation needed.

In [3]:
with TemporaryDirectory() as tmp:
    ratings = movielens.load_pandas_df(size=MOVIELENS_DATA_SIZE, local_cache_path=tmp)
    items = movielens.load_item_df(
        size=MOVIELENS_DATA_SIZE, local_cache_path=tmp, genres_col="genre", year_col="release_year"
    )

# Extract primary genre (first listed) and keep relevant item columns
items["main_genre"] = items["genre"].str.split("|").str[0]
items["release_year"] = items["release_year"].astype(float)
items = items[[ITEM_COL, "main_genre", "release_year"]]

# Per-user chronological split: train 80% / valid 10% / test 10%
train_raw, valid_raw, test_raw = python_chrono_split(
    ratings,
    ratio=[0.8, 0.1, 0.1],
    col_user=USER_COL,
    col_item=ITEM_COL,
)

# Compute user & item statistics from training data only (no label leakage)
user_stats = (
    train_raw.groupby(USER_COL)[RATING_COL]
    .agg(user_mean_rating="mean", user_rating_count="count", user_rating_std="std")
    .reset_index()
)
item_stats = (
    train_raw.groupby(ITEM_COL)[RATING_COL]
    .agg(item_mean_rating="mean", item_rating_count="count", item_rating_std="std")
    .reset_index()
)
global_mean = train_raw[RATING_COL].mean()

# User-genre stats require genre to be available in train first
train_with_genre = train_raw.merge(items[[ITEM_COL, "main_genre"]], on=ITEM_COL, how="left")
user_genre_stats = (
    train_with_genre.groupby([USER_COL, "main_genre"])[RATING_COL]
    .agg(user_genre_mean_rating="mean", user_genre_count="count")
    .reset_index()
)

# Median release year used to impute missing years (a few titles lack a year)
year_median = items["release_year"].median()


def enrich(df: pd.DataFrame) -> pd.DataFrame:
    df = df.merge(user_stats, on=USER_COL, how="left")
    df = df.merge(item_stats, on=ITEM_COL, how="left")
    df = df.merge(items, on=ITEM_COL, how="left")
    df = df.merge(user_genre_stats, on=[USER_COL, "main_genre"], how="left")
    df["user_mean_rating"] = df["user_mean_rating"].fillna(global_mean)
    df["user_rating_count"] = df["user_rating_count"].fillna(0)
    df["user_rating_std"] = df["user_rating_std"].fillna(0)
    df["item_mean_rating"] = df["item_mean_rating"].fillna(global_mean)
    df["item_rating_count"] = df["item_rating_count"].fillna(0)
    df["item_rating_std"] = df["item_rating_std"].fillna(0)
    df["user_genre_mean_rating"] = df["user_genre_mean_rating"].fillna(global_mean)
    df["user_genre_count"] = df["user_genre_count"].fillna(0)
    df["release_year"] = df["release_year"].fillna(year_median)
    df["main_genre"] = df["main_genre"].fillna("Unknown")
    return df


train_raw = enrich(train_raw)
valid_raw = enrich(valid_raw)
test_raw = enrich(test_raw)

# Binarize: rating > user's personal mean → 1 (above-average for this user), else → 0
for df in [train_raw, valid_raw, test_raw]:
    df[LABEL_COL] = (df[RATING_COL] > df["user_mean_rating"]).astype(int)

# Add negatives to training and validation so lambdarank has a meaningful signal
def _neg_samples(df: pd.DataFrame, n_neg: int) -> pd.DataFrame:
    combined = negative_feedback_sampler(
        df, col_user=USER_COL, col_item=ITEM_COL,
        col_feedback=LABEL_COL, n_neg_per_user=n_neg, seed=SEED,
    )
    neg = enrich(combined.loc[combined[LABEL_COL] == 0, [USER_COL, ITEM_COL, LABEL_COL]])
    for col in df.columns:
        if col not in neg.columns:
            neg[col] = 0
    return neg[df.columns]


train_raw = pd.concat([train_raw, _neg_samples(train_raw, N_NEG_TRAIN)], ignore_index=True)
valid_raw = pd.concat([valid_raw, _neg_samples(valid_raw, N_NEG_VALID)], ignore_index=True)


def make_groups(df: pd.DataFrame):
    df = df.sort_values(USER_COL).reset_index(drop=True)
    group_sizes = df.groupby(USER_COL).size().values
    return df, group_sizes


train_data, train_group = make_groups(train_raw)
valid_data, valid_group = make_groups(valid_raw)
test_data, test_group = make_groups(test_raw)

print(f"train: {len(train_data):,}  valid: {len(valid_data):,}  test: {len(test_data):,}")
print(
    f"train queries: {len(train_group):,}  "
    f"valid: {len(valid_group):,}  "
    f"test: {len(test_group):,}"
)
print(
    f"positive rate — train: {train_data[LABEL_COL].mean():.2%}  "
    f"valid: {valid_data[LABEL_COL].mean():.2%}  "
    f"test: {test_data[LABEL_COL].mean():.2%}"
)

100%|██████████| 64.0k/64.0k [00:06<00:00, 10.1kKB/s]


train: 14,988,236  valid: 7,987,755  test: 999,663
train queries: 69,878  valid: 69,878  test: 69,878
positive rate — train: 28.63%  valid: 6.24%  test: 50.63%


## 2. Feature Encoding (NumEncoder)

`NumEncoder` converts all categorical columns into dense numerical features through three steps:
1. **Low-frequency filtering** — rare categories become `<LESS>`, missing values become `<UNK>`.
2. **Sequential target & count encoding** — encodes each sample using statistics from previous samples only (no label leakage). Target statistics are computed against binary labels (0/1).
3. **Binary encoding** — the ordinal-encoded category values are expanded into bit vectors.

In [4]:
encoder = NumEncoder(CATE_COLS, NUME_COLS, LABEL_COL)

train_x, train_y = encoder.fit_transform(train_data)
valid_x, valid_y = encoder.transform(valid_data)
test_x, test_y = encoder.transform(test_data)

# Append main_genre as a native LightGBM categorical feature.
# NumEncoder already uses main_genre for binary/target encoding;
# adding it again as an ordinal code lets LightGBM apply its
# optimal categorical split algorithm directly on the genre.
genres = sorted(train_data["main_genre"].dropna().unique())
genre_map = {g: i for i, g in enumerate(genres)}
CAT_FEATURE_IDX = train_x.shape[1]  # index of the appended genre column


def append_genre(x: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    codes = df["main_genre"].map(genre_map).fillna(-1).astype(int).values.reshape(-1, 1)
    return np.hstack([x, codes])


train_x = append_genre(train_x, train_data)
valid_x = append_genre(valid_x, valid_data)
test_x  = append_genre(test_x,  test_data)

print(f"train_x: {train_x.shape}  valid_x: {valid_x.shape}  test_x: {test_x.shape}")
print(f"label range — train: [{int(train_y.min())}, {int(train_y.max())}]")
print(f"genre categorical feature index: {CAT_FEATURE_IDX}")

2026-03-25 10:24:33,865 [INFO] Filtering and fillna features
100%|██████████| 9/9 [00:00<00:00, 65.93it/s]
2026-03-25 10:24:35,541 [INFO] Ordinal encoding cate features
2026-03-25 10:24:37,748 [INFO] Target encoding cate features
100%|██████████| 1/1 [00:06<00:00,  6.75s/it]
2026-03-25 10:24:44,494 [INFO] Start manual binary encoding
100%|██████████| 1/1 [00:00<00:00,  1.87it/s]
2026-03-25 10:24:48,444 [INFO] Filtering and fillna features
100%|██████████| 9/9 [00:00<00:00, 308.00it/s]
2026-03-25 10:24:49,299 [INFO] Ordinal encoding cate features
2026-03-25 10:24:49,807 [INFO] Target encoding cate features
100%|██████████| 1/1 [00:02<00:00,  2.20s/it]
2026-03-25 10:24:52,004 [INFO] Start manual binary encoding
100%|██████████| 1/1 [00:00<00:00,  3.94it/s]
2026-03-25 10:24:53,411 [INFO] Filtering and fillna features
100%|██████████| 9/9 [00:00<00:00, 1206.57it/s]
2026-03-25 10:24:53,527 [INFO] Ordinal encoding cate features
2026-03-25 10:24:53,657 [INFO] Target encoding cate features
100

train_x: (14988236, 17)  valid_x: (7987755, 17)  test_x: (999663, 17)
label range — train: [0, 1]
genre categorical feature index: 16


## 3. Training (lgb.LGBMRanker.fit)

Pass `train_group` and `valid_group` so LightGBM knows which items belong to the same query. Early stopping monitors NDCG on the validation set.

In [5]:
ranker = LGBMRanker(n_estimators=NUM_BOOST_ROUND, **PARAMS)

In [6]:
ranker.fit(
    train_x,
    train_y.reshape(-1),
    group=train_group,
    eval_set=[(valid_x, valid_y.reshape(-1))],
    eval_group=[valid_group],
    categorical_feature=[CAT_FEATURE_IDX],
    eval_at=[5, 10],
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(EARLY_STOPPING_ROUNDS)],
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.249441 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2674
[LightGBM] [Info] Number of data points in the train set: 14988236, number of used features: 17
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@5: 0.431109	valid_0's ndcg@10: 0.479554
[200]	valid_0's ndcg@5: 0.435574	valid_0's ndcg@10: 0.48368
[300]	valid_0's ndcg@5: 0.432895	valid_0's ndcg@10: 0.48111
Early stopping, best iteration is:
[214]	valid_0's ndcg@5: 0.436161	valid_0's ndcg@10: 0.484359


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.01
,n_estimators,10000
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## 4. Evaluation Negative Sampling

Sample `N_NEG_EVAL` unobserved items per test user and add them to the evaluation pool so the model must rank relevant items above many irrelevant candidates — the actual recommendation task.

> **Note**: training and validation negatives (`N_NEG_TRAIN` / `N_NEG_VALID`) are added during data preparation (section 1), before encoding and model fitting.

In [7]:
_neg_eval = negative_feedback_sampler(
    test_data, col_user=USER_COL, col_item=ITEM_COL,
    col_feedback=LABEL_COL, n_neg_per_user=N_NEG_EVAL, seed=SEED,
)
neg_df = enrich(_neg_eval.loc[_neg_eval[LABEL_COL] == 0, [USER_COL, ITEM_COL, LABEL_COL]])
for col in test_data.columns:
    if col not in neg_df.columns:
        neg_df[col] = 0
neg_df = neg_df[test_data.columns]
neg_x, _ = encoder.transform(neg_df)
neg_x = append_genre(neg_x, neg_df)
neg_scores = ranker.predict(neg_x)

print(f"Evaluation negative pool: {len(neg_df):,}  ({N_NEG_EVAL} per user)")

2026-03-25 10:28:39,910 [INFO] Filtering and fillna features
100%|██████████| 9/9 [00:00<00:00, 285.95it/s]
2026-03-25 10:28:40,679 [INFO] Ordinal encoding cate features
2026-03-25 10:28:41,144 [INFO] Target encoding cate features
100%|██████████| 1/1 [00:01<00:00,  1.92s/it]
2026-03-25 10:28:43,066 [INFO] Start manual binary encoding
100%|██████████| 1/1 [00:00<00:00,  4.12it/s]
/Users/wooklee/.venvs/recommenders/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


Evaluation negative pool: 6,987,800  (100 per user)


## 5. Inference and Evaluation

Score all test items and compute **NDCG@5** and **NDCG@10** over the combined pool of observed interactions (binary labels) and sampled negatives (label = 0).

In [8]:
scores = ranker.predict(test_x)

# rating_true: only relevant items (label=1) — what the user actually liked
true_df = (
    test_data[test_data[LABEL_COL] == 1][[USER_COL, ITEM_COL]]
    .assign(**{RATING_COL: 1})
    .reset_index(drop=True)
)

# rating_pred: full candidate pool (observed + sampled negatives) with model scores
pred_df = pd.concat(
    [
        test_data[[USER_COL, ITEM_COL]].assign(prediction=scores),
        neg_df[[USER_COL, ITEM_COL]].assign(prediction=neg_scores),
    ]
).reset_index(drop=True)

/Users/wooklee/.venvs/recommenders/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


In [9]:
eval_map10 = map(
    true_df,
    pred_df,
    col_user=USER_COL,
    col_item=ITEM_COL,
    col_rating=RATING_COL,
    col_prediction="prediction",
    k=10,
)
eval_ndcg10 = ndcg_at_k(
    true_df,
    pred_df,
    col_user=USER_COL,
    col_item=ITEM_COL,
    col_rating=RATING_COL,
    col_prediction="prediction",
    k=10,
    score_type="binary",
)

print(f"MAP@10:  {eval_map10:.4f}")
print(f"NDCG@10: {eval_ndcg10:.4f}")

MAP@10:  0.2625
NDCG@10: 0.4228


In [10]:
# Record results for tests - ignore this cell
store_metadata("map_at_10", eval_map10)
store_metadata("ndcg_at_10", eval_ndcg10)

## 6. Top-K Ranking Example

In a real recommendation pipeline, scores are computed per query (user session) and items are sorted by descending score. Below we display the top-5 items for the first few test queries. Labels are binary (1 = relevant, 0 = irrelevant).

In [15]:
eval_pool = pd.concat(
    [
        test_data[[USER_COL, ITEM_COL, LABEL_COL]].assign(score=scores),
        neg_df[[USER_COL, ITEM_COL, LABEL_COL]].assign(score=neg_scores),
    ]
).reset_index(drop=True)

shown = 0
for user in test_data[USER_COL].unique():
    if shown >= 3:
        break
    query_df = eval_pool[eval_pool[USER_COL] == user][[ITEM_COL, "score", LABEL_COL]]
    query_df = query_df.rename(columns={LABEL_COL: "relevant"})
    top10 = query_df.nlargest(10, "score").reset_index(drop=True)
    # only show users where at least one relevant item appears in top-10
    if top10["relevant"].sum() == 0:
        continue
    n_pos = int((query_df["relevant"] == 1).sum())
    print(f"User {user} (pool: {len(query_df)}, relevant: {n_pos})  top-10:")
    display(top10)
    shown += 1


User 4 (pool: 104, relevant: 1)  top-10:


,itemID,score,relevant
0,47629,0.304680,0
1,3159,0.172213,0
2,2384,0.160964,0
3,59810,-0.020101,0
4,62644,-0.024739,0
5,54094,-0.032849,0
6,266,-0.087349,1
7,37855,-0.109719,0
8,432,-0.165482,0
9,440,-0.182889,0


User 5 (pool: 109, relevant: 8)  top-10:


,itemID,score,relevant
0,1258,1.932562,1
1,1997,1.607159,0
2,508,0.587137,0
3,48774,0.037463,0
4,3176,0.031328,0
5,1199,-0.020000,1
6,969,-0.103166,1
7,926,-0.372301,1
8,2712,-0.423829,0
9,1260,-0.442415,0


User 6 (pool: 104, relevant: 3)  top-10:


,itemID,score,relevant
0,1061,1.287659,0
1,1580,0.520161,1
2,2797,0.433852,0
3,1573,0.269185,0
4,52042,0.192651,0
5,2628,0.016006,1
6,51080,0.011578,0
7,40955,-0.219391,0
8,168,-0.381675,0
9,49274,-0.684797,0


## 7. Save and Load

The underlying booster can be saved with `booster_.save_model()` and restored with `lgb.Booster` for inference, producing identical predictions.

In [14]:
with TemporaryDirectory() as tmp:
    model_path = os.path.join(tmp, "ranker.lgb")

    ranker.booster_.save_model(model_path)

    booster = lgb.Booster(model_file=model_path)
    loaded_scores = booster.predict(test_x)

assert np.allclose(scores, loaded_scores), "Predictions differ after save/load!"
print("Save/load verified: predictions are identical.")

Save/load verified: predictions are identical.
